In [7]:
import pandas as pd
import os
from zipfile import ZipFile
import requests

# ----------------------------
# 1. Download & Extract MovieLens 1M
# ----------------------------
dataset_url = "https://files.grouplens.org/datasets/movielens/ml-1m.zip"
dataset_zip = "ml-1m.zip"
extracted_path = "ml-1m"

if not os.path.exists(extracted_path):
    print("Downloading MovieLens 1M dataset...")
    r = requests.get(dataset_url)
    with open(dataset_zip, "wb") as f:
        f.write(r.content)
    with ZipFile(dataset_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Download and extraction done.")

# ----------------------------
# 2. Load all files
# ----------------------------
ratings_file = os.path.join(extracted_path, "ratings.dat")
movies_file = os.path.join(extracted_path, "movies.dat")
users_file  = os.path.join(extracted_path, "users.dat")

ratings = pd.read_csv(
    ratings_file, sep="::", engine="python",
    names=["UserID","MovieID","Rating","Timestamp"]
)
movies = pd.read_csv(
    movies_file, sep="::", engine="python",
    names=["MovieID","Title","Genres"], encoding="latin-1"
)
users = pd.read_csv(
    users_file, sep="::", engine="python",
    names=["UserID","Gender","Age","Occupation","Zip-code"]
)

# ----------------------------
# 3. Convert timestamp
# ----------------------------
ratings['Timestamp'] = pd.to_datetime(ratings['Timestamp'], unit='s')

# ----------------------------
# 4. Check for duplicates
# ----------------------------
print("Duplicate rows:")
print(f"Ratings: {ratings.duplicated().sum()}")
print(f"Movies:  {movies.duplicated().sum()}")
print(f"Users:   {users.duplicated().sum()}")

# Drop duplicates
ratings = ratings.drop_duplicates().reset_index(drop=True)
movies  = movies.drop_duplicates().reset_index(drop=True)
users   = users.drop_duplicates().reset_index(drop=True)

# ----------------------------
# 5. Check for missing values in detail
# ----------------------------
def report_missing(df, name):
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"No missing values in {name}.")
    else:
        print(f"Missing values in {name}:")
        for col in df.columns:
            missing_count = df[col].isnull().sum()
            if missing_count > 0:
                print(f" - Column '{col}': {missing_count} missing")
        print("Rows with missing values:")
        print(df[df.isnull().any(axis=1)])

report_missing(ratings, "ratings")
report_missing(movies, "movies")
report_missing(users, "users")

# ----------------------------
# 6. Filter ratings to May → Dec 2000
# ----------------------------
start_date = '2000-05-01'
end_date   = '2000-12-31'
ratings = ratings[(ratings['Timestamp'] >= start_date) & (ratings['Timestamp'] <= end_date)].reset_index(drop=True)
print("\nRatings after filtering May → Dec 2000:", ratings.shape)


Duplicate rows:
Ratings: 0
Movies:  0
Users:   0
No missing values in ratings.
No missing values in movies.
No missing values in users.

Ratings after filtering May → Dec 2000: (891189, 4)


In [8]:
# ----------------------------
# Create monthly slices
# ----------------------------
monthly_bins = pd.date_range(start=start_date, end='2001-01-01', freq='MS')
ratings['SliceNum'] = pd.cut(ratings['Timestamp'], bins=monthly_bins, right=False).cat.codes + 1

# ----------------------------
# Inspect each slice
# ----------------------------
slice_summary = []

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    num_ratings = slice_df.shape[0]
    num_movies  = slice_df['MovieID'].nunique()
    num_users   = slice_df['UserID'].nunique()
    slice_summary.append({
        'SliceNum': s,
        'Ratings': num_ratings,
        'UniqueMovies': num_movies,
        'UniqueUsers': num_users
    })

slice_summary_df = pd.DataFrame(slice_summary)
print(slice_summary_df)


   SliceNum  Ratings  UniqueMovies  UniqueUsers
0         1    67437          2920          486
1         2    54486          2913          508
2         3    90334          3135          778
3         4   182109          3298         1310
4         5    52421          3083          576
5         6    42294          2993          500
6         7   290793          3552         2357
7         8   111315          3331         1215


In [9]:
import os
import pandas as pd

# ----------------------------
# 1. Create folder for slice metadata
# ----------------------------
os.makedirs("slice_metadata", exist_ok=True)

# ----------------------------
# 2. Split movies into popularity tiers per slice
# ----------------------------
high_pct, mid_pct, low_pct = 0.2, 0.3, 0.5  # 20/30/50 split

for s in sorted(ratings['SliceNum'].unique()):
    slice_df = ratings[ratings['SliceNum'] == s]
    movie_counts = slice_df.groupby('MovieID')['Rating'].count().sort_values(ascending=False)
    total_movies = len(movie_counts)
    
    # Determine cutoff indices
    high_cut  = int(total_movies * high_pct)
    mid_cut   = high_cut + int(total_movies * mid_pct)
    
    # Assign tiers
    movie_tiers = pd.DataFrame({'MovieID': movie_counts.index})
    movie_tiers['PopTier'] = 'low'  # default
    movie_tiers.loc[:high_cut-1, 'PopTier'] = 'high'
    movie_tiers.loc[high_cut:mid_cut-1, 'PopTier'] = 'medium'
    
    # Save slice metadata
    movie_tiers.to_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv", index=False)
    print(f"Slice {s}: {total_movies} movies → High:{high_cut}, Medium:{mid_cut-high_cut}, Low:{total_movies-mid_cut}")


Slice 1: 2920 movies → High:584, Medium:876, Low:1460
Slice 2: 2913 movies → High:582, Medium:873, Low:1458
Slice 3: 3135 movies → High:627, Medium:940, Low:1568
Slice 4: 3298 movies → High:659, Medium:989, Low:1650
Slice 5: 3083 movies → High:616, Medium:924, Low:1543
Slice 6: 2993 movies → High:598, Medium:897, Low:1498
Slice 7: 3552 movies → High:710, Medium:1065, Low:1777
Slice 8: 3331 movies → High:666, Medium:999, Low:1666


In [10]:
# ----------------------------
# User type assignment per slice
# ----------------------------
import os
import pandas as pd

slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata", exist_ok=True)

threshold = 0.7  # HighProportion threshold to classify mainstream

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty: 
        continue

    # Load movie popularity tiers for this slice
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    movie_tier_map = dict(zip(movie_tiers.MovieID, movie_tiers.PopTier))

    # Map each rating to movie tier
    slice_df['Tier'] = slice_df['MovieID'].map(movie_tier_map)

    # Count number of ratings per tier per user
    tier_counts = slice_df.groupby(['UserID', 'Tier']).size().unstack(fill_value=0)
    # Ensure all columns exist
    for col in ['high', 'medium', 'low']:
        if col not in tier_counts.columns:
            tier_counts[col] = 0

    # Compute HighProportion
    tier_counts['Total'] = tier_counts['high'] + tier_counts['medium'] + tier_counts['low']
    tier_counts['HighProportion'] = tier_counts['high'] / tier_counts['Total']

    # Assign user type
    tier_counts['UserType'] = tier_counts['HighProportion'].apply(
        lambda x: 'mainstream' if x >= threshold else 'niche'
    )

    # Save result
    user_type_df = tier_counts[['UserType']].reset_index()
    user_type_df.to_csv(f"slice_metadata/user_labels_slice_{s}.csv", index=False)
    
    print(f"Slice {s}: Total Users = {len(user_type_df)}, Mainstream = {sum(user_type_df.UserType=='mainstream')}, Niche = {sum(user_type_df.UserType=='niche')}")


Slice 1: Total Users = 486, Mainstream = 233, Niche = 253
Slice 2: Total Users = 508, Mainstream = 239, Niche = 269
Slice 3: Total Users = 778, Mainstream = 332, Niche = 446
Slice 4: Total Users = 1310, Mainstream = 598, Niche = 712
Slice 5: Total Users = 576, Mainstream = 276, Niche = 300
Slice 6: Total Users = 500, Mainstream = 223, Niche = 277
Slice 7: Total Users = 2357, Mainstream = 1420, Niche = 937
Slice 8: Total Users = 1215, Mainstream = 596, Niche = 619


In [11]:
import pandas as pd
import os
import torch

# ----------------------------
# 1. KG construction per slice with relation types and mappings
# ----------------------------
slice_nums = sorted(ratings['SliceNum'].unique())
os.makedirs("slice_metadata/kg", exist_ok=True)

# Define relation types
REL_USER_MOVIE = 0
REL_MOVIE_GENRE = 1
REL_MOVIE_POPTIER = 2
REL_USER_USERTYPE = 3

for s in slice_nums:
    slice_df = ratings[ratings['SliceNum'] == s].copy()
    if slice_df.empty:
        continue
    
    print(f"\nConstructing KG for slice {s}...")

    # Load slice-specific metadata
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv")
    user_types  = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv")
    
    # Extract unique nodes
    users = slice_df['UserID'].unique()
    movies_slice = slice_df['MovieID'].unique()
    genres = set()
    for g in movies['Genres']:
        genres.update(g.split('|'))
    genres = list(genres)
    
    pop_tiers = ['high', 'medium', 'low']
    utypes = ['mainstream', 'niche']
    
    # ----------------------------
    # Slice-local IDs
    user2nid = {u: i for i, u in enumerate(users)}
    movie2nid = {m: i + len(users) for i, m in enumerate(movies_slice)}
    genre2nid = {g: i + len(users) + len(movies_slice) for i, g in enumerate(genres)}
    pop2nid = {p: i + len(users) + len(movies_slice) + len(genres) for i, p in enumerate(pop_tiers)}
    utype2nid = {ut: i + len(users) + len(movies_slice) + len(genres) + len(pop_tiers) for i, ut in enumerate(utypes)}
    
    # Save mapping tables for hybrid warm-start later
    pd.DataFrame(list(user2nid.items()), columns=['UserID','NodeID']).to_csv(f"slice_metadata/kg/user_map_slice_{s}.csv", index=False)
    pd.DataFrame(list(movie2nid.items()), columns=['MovieID','NodeID']).to_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv", index=False)
    
    # ----------------------------
    # Build edges with relation types
    edges = []

    # User ↔ Movie
    for row in slice_df.itertuples():
        uid, mid = user2nid[row.UserID], movie2nid[row.MovieID]
        edges.append((uid, mid, REL_USER_MOVIE))
        edges.append((mid, uid, REL_USER_MOVIE))  # bi-directional

    # Movie ↔ Genre
    for row in movies.itertuples():
        if row.MovieID not in movie2nid:
            continue
        mid = movie2nid[row.MovieID]
        for g in row.Genres.split('|'):
            edges.append((mid, genre2nid[g], REL_MOVIE_GENRE))
            edges.append((genre2nid[g], mid, REL_MOVIE_GENRE))

    # Movie ↔ PopTier
    for row in movie_tiers.itertuples():
        mid = movie2nid[row.MovieID]
        edges.append((mid, pop2nid[row.PopTier], REL_MOVIE_POPTIER))
        edges.append((pop2nid[row.PopTier], mid, REL_MOVIE_POPTIER))

    # User ↔ UserType
    for row in user_types.itertuples():
        uid = user2nid[row.UserID]
        edges.append((uid, utype2nid[row.UserType], REL_USER_USERTYPE))
        edges.append((utype2nid[row.UserType], uid, REL_USER_USERTYPE))
    
    # Save edges with relation types
    edges_df = pd.DataFrame(edges, columns=['Src','Dst','RelType'])
    edges_df.to_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv", index=False)
    
    print(f"Slice {s} KG: {len(edges_df)} edges, {len(user2nid)} users, {len(movie2nid)} movies, {len(genre2nid)} genres, {len(pop2nid)} pop tiers, {len(utype2nid)} user types")



Constructing KG for slice 1...
Slice 1 KG: 151890 edges, 486 users, 2920 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 2...
Slice 2 KG: 126078 edges, 508 users, 2913 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 3...
Slice 3 KG: 199352 edges, 778 users, 3135 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 4...
Slice 4 KG: 384750 edges, 1310 users, 3298 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 5...
Slice 5 KG: 122904 edges, 576 users, 3083 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 6...
Slice 6 KG: 102034 edges, 500 users, 2993 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 7...
Slice 7 KG: 605404 edges, 2357 users, 3552 movies, 18 genres, 3 pop tiers, 2 user types

Constructing KG for slice 8...
Slice 8 KG: 243150 edges, 1215 users, 3331 movies, 18 genres, 3 pop tiers, 2 user types


PROJECT README
Temporal Knowledge Graphs for Recommender Systems
================================================

1. What this project is about
-----------------------------

This project studies how user preferences and item popularity change over time,
and how recommender models can adapt to those changes instead of treating data as static.

Most recommender systems:
- Merge all interactions into one dataset
- Assume user behavior and item popularity are fixed
- Ignore temporal structure

That is unrealistic.

In this project, we:
- Split user–item interactions into time slices
- Build a separate Knowledge Graph (KG) for each slice
- Train models slice by slice
- Compare a baseline recommender with a more expressive final model

The goal is to evaluate whether modeling structure and time actually improves recommendations.


2. Why use a Knowledge Graph
----------------------------

A standard user–item matrix only captures:
“User A interacted with Movie B”.

A Knowledge Graph allows us to represent richer structure:
- Users have behavioral types (mainstream vs niche)
- Movies have genres
- Movies have popularity levels
- All of these can change over time

By encoding interactions as a graph:
- Simple graph models can be used as baselines
- Relation-aware models can exploit edge types
- The same data supports multiple model families


3. Why time slicing
-------------------

User behavior in May 2000 is not the same as in December 2000.

Instead of building one global graph, we:
- Split ratings into monthly time slices
- Build one KG per slice
- Allow users and movies to appear, disappear, or change context

This enables:
- Temporal evaluation
- Embedding warm-starting
- Studying preference drift


4. Models trained later
-----------------------

This preprocessing supports two categories of models:

1. Baseline model
   - Treats all edges as the same
   - Uses only graph structure
   - Ignores relation types

2. Final model
   - Uses relation types
   - Distinguishes different kinds of edges
   - Exploits richer KG structure

Both models use the same preprocessed data.


5. Dataset
----------

- MovieLens 1M
- Ratings filtered to May 2000 – December 2000
- Monthly time slices


6. Outputs per time slice
------------------------

For each slice, the following are created:
- Movie popularity tiers
- User behavioral types
- Slice-specific Knowledge Graph
- Node ID mappings for temporal alignment

All outputs are stored as CSV files.


7. Directory structure
----------------------

.
├── slice_metadata/
│   ├── movie_pop_tiers_slice_1.csv
│   ├── user_labels_slice_1.csv
│   ├── ...
│   └── kg/
│       ├── kg_edges_slice_1.csv
│       ├── user_map_slice_1.csv
│       ├── movie_map_slice_1.csv
│       └── ...


8. Movie popularity tiers
-------------------------

Files:
slice_metadata/movie_pop_tiers_slice_{s}.csv

Description:
- Movies are ranked by number of ratings within a slice
- Assigned to popularity tiers:
  - high
  - medium
  - low

These tiers capture short-term popularity, not global popularity.


9. User behavioral types
------------------------

Files:
slice_metadata/user_labels_slice_{s}.csv

Description:
- Users are labeled based on what they consume in that slice
- Users who mostly rate popular movies → mainstream
- Others → niche

User types can change across slices.


10. Knowledge Graphs
--------------------

Each slice has its own Knowledge Graph.


10.1 Edge list
--------------

Files:
slice_metadata/kg/kg_edges_slice_{s}.csv

Columns:
- Src     : source node ID
- Dst     : destination node ID
- RelType : relation type ID

All edges are stored as bidirectional.


10.2 Relation types
-------------------

Relation ID meanings:

0 → User ↔ Movie  
1 → Movie ↔ Genre  
2 → Movie ↔ Popularity Tier  
3 → User ↔ User Type  

Baseline models ignore RelType.
Final models use RelType.


11. Node IDs
------------

Node IDs are:
- Integers
- Unique within a slice
- Assigned in non-overlapping ranges by node type

This guarantees that users, movies, genres, tiers, and user types
cannot be confused with each other.


12. Node ID mappings (important)
--------------------------------

Files:
- slice_metadata/kg/user_map_slice_{s}.csv
- slice_metadata/kg/movie_map_slice_{s}.csv

These map original UserID / MovieID to slice-local node IDs.

They are required for:
- Matching entities across slices
- Warm-starting embeddings
- Temporal stabilization during training


13. What is NOT stored
----------------------

- No trained models
- No tensors
- No pickled Python objects
- No framework-specific formats

Everything is reconstructed from CSVs in training notebooks.


14. How a new training session should start
-------------------------------------------

A fresh session should:
1. Select a slice
2. Load KG edges and mappings
3. Convert data to tensors as needed
4. Train baseline and final models
5. Optionally align embeddings across slices


15. Summary
-----------

This project builds time-aware Knowledge Graphs from MovieLens data
to compare a baseline recommender against a relation-aware model
while explicitly modeling temporal dynamics.


K = 10

In [12]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=10):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: Multi-Relational DistMult (Baseline)
# ----------------------------
class DistMultModel(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.rel_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def forward(self, adj_dict=None):
        h = self.node_emb.weight
        r_0 = self.rel_emb.weight[0] 
        return h * r_0

    def get_triple_score(self, h_idx, r_idx, t_idx):
        h = self.node_emb(h_idx)
        r = self.rel_emb(r_idx)
        t = self.node_emb(t_idx)
        return (h * r * t).sum(dim=1)

def build_relational_adjacency(kg_edges_df, num_nodes):
    return {}

# ----------------------------
# 4. Training Loop (Baseline: No Fairness Retries)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
KG_LAMBDA = 0.1 

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())
for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    kg_h = torch.LongTensor(kg_edges_df['Src'].values).to(device)
    kg_r = torch.LongTensor(kg_edges_df['RelType'].values).to(device)
    kg_t = torch.LongTensor(kg_edges_df['Dst'].values).to(device)
    num_kg_triples = len(kg_h)

    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    # BASELINE: No weighting based on tiers or user types
    # All weights are set to 1.0
    weight_matrix = torch.ones((2, 3), dtype=torch.float, device=device)
    
    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        item_tier_tensor[node_id] = tier_idx_map.get(movie_tiers.get(mid, 'low'), 0)

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        user_type_tensor[node_id] = u_type_idx_map.get(user_labels.get(uid, 'niche'), 1)

    # ----------------------------
    # WARM STARTING
    # ----------------------------
    model = DistMultModel(max_node_id, max_rel_id).to(device)
    if prev_model_state:
        with torch.no_grad():
            prev_nodes = prev_model_state['node_emb.weight'].to(device)
            for entity_key, curr_id in current_node_lookup.items():
                if entity_key in prev_node_map:
                    prev_id = prev_node_map[entity_key]
                    if prev_id < prev_nodes.shape[0]:
                        model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]
            
            if 'rel_emb.weight' in prev_model_state:
                prev_rel = prev_model_state['rel_emb.weight'].to(device)
                sz = min(model.rel_emb.weight.shape[0], prev_rel.shape[0])
                model.rel_emb.weight.data[:sz] = prev_rel[:sz]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
    
    # Single training pass per slice (no retries)
    for epoch in range(EPOCHS):
        rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
        neg_nodes = all_item_nodes[rand_indices]
        kg_neg_nodes = torch.randint(0, max_node_id, (num_train_samples,), device=device)
        
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_nodes[b]
            
            # Recommendation Loss
            final_emb = model(adj_dict)
            pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
            neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
            bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores).mean()

            # KG Loss
            kg_idx = torch.randint(0, num_kg_triples, (len(b),), device=device)
            h_k, r_k, t_k = kg_h[kg_idx], kg_r[kg_idx], kg_t[kg_idx]
            t_neg_k = kg_neg_nodes[b]
            pos_kg = model.get_triple_score(h_k, r_k, t_k)
            neg_kg = model.get_triple_score(h_k, r_k, t_neg_k)
            kg_loss = -torch.nn.functional.logsigmoid(pos_kg - neg_kg).mean()

            loss = bpr + (KG_LAMBDA * kg_loss)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Final Metrics for slice
    metrics, count = compute_metrics(model, adj_dict, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"PopExposure: {metrics['PopExposure']} (Baseline - No Weights)")
        
        # Save state for next slice's warm start
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_node_map = current_node_lookup.copy()

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0391    | 0.1451    | 0.0213    | 0.1536    | 0.9323
PopExposure: {'mainstream': {'low': '0.00%', 'medium': '0.64%', 'high': '99.36%'}, 'niche': {'low': '1.84%', 'medium': '21.84%', 'high': '76.33%'}} (Baseline - No Weights)
2 -> 3        | 117    | 0.0236    | 0.1040    | 0.0179    | 0.1861    | 0.8900
PopExposure: {'mainstream': {'low': '0.00%', 'medium': '4.65%', 'high': '95.35%'}, 'niche': {'low': '5.00%', 'medium': '27.57%', 'high': '67.43%'}} (Baseline - No Weights)
3 -> 4        | 166    | 0.0310    | 0.0688    | 0.0368    | 0.1901    | 0.8216
PopExposure: {'mainstream': {'low': '7.90%', 'medium': '13.39%', 'high': '78.71%'}, 'niche': {'low': '16.44%', 'medium': '33.37%', 'high': '50.19%'}} (Baseline - No Weights)
4 -> 5        | 211    | 0.0197    | 0.

K = 25

In [13]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=25):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: Multi-Relational DistMult (Baseline)
# ----------------------------
class DistMultModel(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.rel_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def forward(self, adj_dict=None):
        h = self.node_emb.weight
        r_0 = self.rel_emb.weight[0] 
        return h * r_0

    def get_triple_score(self, h_idx, r_idx, t_idx):
        h = self.node_emb(h_idx)
        r = self.rel_emb(r_idx)
        t = self.node_emb(t_idx)
        return (h * r * t).sum(dim=1)

def build_relational_adjacency(kg_edges_df, num_nodes):
    return {}

# ----------------------------
# 4. Training Loop (Baseline: No Fairness Retries)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
KG_LAMBDA = 0.1 

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())
for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    kg_h = torch.LongTensor(kg_edges_df['Src'].values).to(device)
    kg_r = torch.LongTensor(kg_edges_df['RelType'].values).to(device)
    kg_t = torch.LongTensor(kg_edges_df['Dst'].values).to(device)
    num_kg_triples = len(kg_h)

    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    # BASELINE: No weighting based on tiers or user types
    # All weights are set to 1.0
    weight_matrix = torch.ones((2, 3), dtype=torch.float, device=device)
    
    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        item_tier_tensor[node_id] = tier_idx_map.get(movie_tiers.get(mid, 'low'), 0)

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        user_type_tensor[node_id] = u_type_idx_map.get(user_labels.get(uid, 'niche'), 1)

    # ----------------------------
    # WARM STARTING
    # ----------------------------
    model = DistMultModel(max_node_id, max_rel_id).to(device)
    if prev_model_state:
        with torch.no_grad():
            prev_nodes = prev_model_state['node_emb.weight'].to(device)
            for entity_key, curr_id in current_node_lookup.items():
                if entity_key in prev_node_map:
                    prev_id = prev_node_map[entity_key]
                    if prev_id < prev_nodes.shape[0]:
                        model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]
            
            if 'rel_emb.weight' in prev_model_state:
                prev_rel = prev_model_state['rel_emb.weight'].to(device)
                sz = min(model.rel_emb.weight.shape[0], prev_rel.shape[0])
                model.rel_emb.weight.data[:sz] = prev_rel[:sz]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
    
    # Single training pass per slice (no retries)
    for epoch in range(EPOCHS):
        rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
        neg_nodes = all_item_nodes[rand_indices]
        kg_neg_nodes = torch.randint(0, max_node_id, (num_train_samples,), device=device)
        
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_nodes[b]
            
            # Recommendation Loss
            final_emb = model(adj_dict)
            pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
            neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
            bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores).mean()

            # KG Loss
            kg_idx = torch.randint(0, num_kg_triples, (len(b),), device=device)
            h_k, r_k, t_k = kg_h[kg_idx], kg_r[kg_idx], kg_t[kg_idx]
            t_neg_k = kg_neg_nodes[b]
            pos_kg = model.get_triple_score(h_k, r_k, t_k)
            neg_kg = model.get_triple_score(h_k, r_k, t_neg_k)
            kg_loss = -torch.nn.functional.logsigmoid(pos_kg - neg_kg).mean()

            loss = bpr + (KG_LAMBDA * kg_loss)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Final Metrics for slice
    metrics, count = compute_metrics(model, adj_dict, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"PopExposure: {metrics['PopExposure']} (Baseline - No Weights)")
        
        # Save state for next slice's warm start
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_node_map = current_node_lookup.copy()

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.0900    | 0.1406    | 0.0495    | 0.1694    | 0.8859
PopExposure: {'mainstream': {'low': '0.00%', 'medium': '1.45%', 'high': '98.55%'}, 'niche': {'low': '3.18%', 'medium': '23.67%', 'high': '73.14%'}} (Baseline - No Weights)
2 -> 3        | 117    | 0.0558    | 0.1041    | 0.0121    | 0.1817    | 0.8343
PopExposure: {'mainstream': {'low': '0.09%', 'medium': '6.33%', 'high': '93.58%'}, 'niche': {'low': '4.97%', 'medium': '28.70%', 'high': '66.32%'}} (Baseline - No Weights)
3 -> 4        | 166    | 0.0518    | 0.0774    | 0.0444    | 0.1626    | 0.7595
PopExposure: {'mainstream': {'low': '6.45%', 'medium': '15.74%', 'high': '77.81%'}, 'niche': {'low': '14.85%', 'medium': '31.73%', 'high': '53.42%'}} (Baseline - No Weights)
4 -> 5        | 211    | 0.0528    | 0.

K = 50

In [14]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=50):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: Multi-Relational DistMult (Baseline)
# ----------------------------
class DistMultModel(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.rel_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def forward(self, adj_dict=None):
        h = self.node_emb.weight
        r_0 = self.rel_emb.weight[0] 
        return h * r_0

    def get_triple_score(self, h_idx, r_idx, t_idx):
        h = self.node_emb(h_idx)
        r = self.rel_emb(r_idx)
        t = self.node_emb(t_idx)
        return (h * r * t).sum(dim=1)

def build_relational_adjacency(kg_edges_df, num_nodes):
    return {}

# ----------------------------
# 4. Training Loop (Baseline: No Fairness Retries)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
KG_LAMBDA = 0.1 

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())
for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    kg_h = torch.LongTensor(kg_edges_df['Src'].values).to(device)
    kg_r = torch.LongTensor(kg_edges_df['RelType'].values).to(device)
    kg_t = torch.LongTensor(kg_edges_df['Dst'].values).to(device)
    num_kg_triples = len(kg_h)

    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    # BASELINE: No weighting based on tiers or user types
    # All weights are set to 1.0
    weight_matrix = torch.ones((2, 3), dtype=torch.float, device=device)
    
    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        item_tier_tensor[node_id] = tier_idx_map.get(movie_tiers.get(mid, 'low'), 0)

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        user_type_tensor[node_id] = u_type_idx_map.get(user_labels.get(uid, 'niche'), 1)

    # ----------------------------
    # WARM STARTING
    # ----------------------------
    model = DistMultModel(max_node_id, max_rel_id).to(device)
    if prev_model_state:
        with torch.no_grad():
            prev_nodes = prev_model_state['node_emb.weight'].to(device)
            for entity_key, curr_id in current_node_lookup.items():
                if entity_key in prev_node_map:
                    prev_id = prev_node_map[entity_key]
                    if prev_id < prev_nodes.shape[0]:
                        model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]
            
            if 'rel_emb.weight' in prev_model_state:
                prev_rel = prev_model_state['rel_emb.weight'].to(device)
                sz = min(model.rel_emb.weight.shape[0], prev_rel.shape[0])
                model.rel_emb.weight.data[:sz] = prev_rel[:sz]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
    
    # Single training pass per slice (no retries)
    for epoch in range(EPOCHS):
        rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
        neg_nodes = all_item_nodes[rand_indices]
        kg_neg_nodes = torch.randint(0, max_node_id, (num_train_samples,), device=device)
        
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_nodes[b]
            
            # Recommendation Loss
            final_emb = model(adj_dict)
            pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
            neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
            bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores).mean()

            # KG Loss
            kg_idx = torch.randint(0, num_kg_triples, (len(b),), device=device)
            h_k, r_k, t_k = kg_h[kg_idx], kg_r[kg_idx], kg_t[kg_idx]
            t_neg_k = kg_neg_nodes[b]
            pos_kg = model.get_triple_score(h_k, r_k, t_k)
            neg_kg = model.get_triple_score(h_k, r_k, t_neg_k)
            kg_loss = -torch.nn.functional.logsigmoid(pos_kg - neg_kg).mean()

            loss = bpr + (KG_LAMBDA * kg_loss)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Final Metrics for slice
    metrics, count = compute_metrics(model, adj_dict, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"PopExposure: {metrics['PopExposure']} (Baseline - No Weights)")
        
        # Save state for next slice's warm start
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_node_map = current_node_lookup.copy()

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1415    | 0.1488    | 0.0582    | 0.1767    | 0.8464
PopExposure: {'mainstream': {'low': '0.04%', 'medium': '2.47%', 'high': '97.49%'}, 'niche': {'low': '4.04%', 'medium': '24.98%', 'high': '70.98%'}} (Baseline - No Weights)
2 -> 3        | 117    | 0.1078    | 0.1131    | 0.0527    | 0.1777    | 0.7904
PopExposure: {'mainstream': {'low': '0.19%', 'medium': '7.53%', 'high': '92.28%'}, 'niche': {'low': '5.65%', 'medium': '28.73%', 'high': '65.62%'}} (Baseline - No Weights)
3 -> 4        | 166    | 0.1043    | 0.0907    | 0.0406    | 0.1463    | 0.7173
PopExposure: {'mainstream': {'low': '5.74%', 'medium': '17.26%', 'high': '77.00%'}, 'niche': {'low': '13.62%', 'medium': '31.33%', 'high': '55.06%'}} (Baseline - No Weights)
4 -> 5        | 211    | 0.1022    | 0.

K = 75

In [15]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=75):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: Multi-Relational DistMult (Baseline)
# ----------------------------
class DistMultModel(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.rel_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def forward(self, adj_dict=None):
        h = self.node_emb.weight
        r_0 = self.rel_emb.weight[0] 
        return h * r_0

    def get_triple_score(self, h_idx, r_idx, t_idx):
        h = self.node_emb(h_idx)
        r = self.rel_emb(r_idx)
        t = self.node_emb(t_idx)
        return (h * r * t).sum(dim=1)

def build_relational_adjacency(kg_edges_df, num_nodes):
    return {}

# ----------------------------
# 4. Training Loop (Baseline: No Fairness Retries)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
KG_LAMBDA = 0.1 

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())
for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    kg_h = torch.LongTensor(kg_edges_df['Src'].values).to(device)
    kg_r = torch.LongTensor(kg_edges_df['RelType'].values).to(device)
    kg_t = torch.LongTensor(kg_edges_df['Dst'].values).to(device)
    num_kg_triples = len(kg_h)

    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    # BASELINE: No weighting based on tiers or user types
    # All weights are set to 1.0
    weight_matrix = torch.ones((2, 3), dtype=torch.float, device=device)
    
    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        item_tier_tensor[node_id] = tier_idx_map.get(movie_tiers.get(mid, 'low'), 0)

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        user_type_tensor[node_id] = u_type_idx_map.get(user_labels.get(uid, 'niche'), 1)

    # ----------------------------
    # WARM STARTING
    # ----------------------------
    model = DistMultModel(max_node_id, max_rel_id).to(device)
    if prev_model_state:
        with torch.no_grad():
            prev_nodes = prev_model_state['node_emb.weight'].to(device)
            for entity_key, curr_id in current_node_lookup.items():
                if entity_key in prev_node_map:
                    prev_id = prev_node_map[entity_key]
                    if prev_id < prev_nodes.shape[0]:
                        model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]
            
            if 'rel_emb.weight' in prev_model_state:
                prev_rel = prev_model_state['rel_emb.weight'].to(device)
                sz = min(model.rel_emb.weight.shape[0], prev_rel.shape[0])
                model.rel_emb.weight.data[:sz] = prev_rel[:sz]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
    
    # Single training pass per slice (no retries)
    for epoch in range(EPOCHS):
        rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
        neg_nodes = all_item_nodes[rand_indices]
        kg_neg_nodes = torch.randint(0, max_node_id, (num_train_samples,), device=device)
        
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_nodes[b]
            
            # Recommendation Loss
            final_emb = model(adj_dict)
            pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
            neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
            bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores).mean()

            # KG Loss
            kg_idx = torch.randint(0, num_kg_triples, (len(b),), device=device)
            h_k, r_k, t_k = kg_h[kg_idx], kg_r[kg_idx], kg_t[kg_idx]
            t_neg_k = kg_neg_nodes[b]
            pos_kg = model.get_triple_score(h_k, r_k, t_k)
            neg_kg = model.get_triple_score(h_k, r_k, t_neg_k)
            kg_loss = -torch.nn.functional.logsigmoid(pos_kg - neg_kg).mean()

            loss = bpr + (KG_LAMBDA * kg_loss)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Final Metrics for slice
    metrics, count = compute_metrics(model, adj_dict, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"PopExposure: {metrics['PopExposure']} (Baseline - No Weights)")
        
        # Save state for next slice's warm start
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_node_map = current_node_lookup.copy()

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.1962    | 0.1654    | 0.0621    | 0.1742    | 0.8208
PopExposure: {'mainstream': {'low': '0.11%', 'medium': '4.34%', 'high': '95.55%'}, 'niche': {'low': '4.52%', 'medium': '26.07%', 'high': '69.41%'}} (Baseline - No Weights)
2 -> 3        | 117    | 0.1570    | 0.1241    | 0.0456    | 0.1700    | 0.7669
PopExposure: {'mainstream': {'low': '0.37%', 'medium': '8.78%', 'high': '90.85%'}, 'niche': {'low': '6.16%', 'medium': '28.49%', 'high': '65.35%'}} (Baseline - No Weights)
3 -> 4        | 166    | 0.1505    | 0.1037    | 0.0353    | 0.1417    | 0.6938
PopExposure: {'mainstream': {'low': '5.55%', 'medium': '17.83%', 'high': '76.62%'}, 'niche': {'low': '13.54%', 'medium': '31.09%', 'high': '55.37%'}} (Baseline - No Weights)
4 -> 5        | 211    | 0.1489    | 0.

K = 100

In [16]:
import os
# CRITICAL FIX: Deterministic behavior for reproducibility
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
import scipy.sparse as sp

# ----------------------------
# 1. Device & Setup
# ----------------------------
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
np.random.seed(42)
torch.use_deterministic_algorithms(True)

# ----------------------------
# 2. Metric Helpers (Unchanged)
# ----------------------------
def get_gini(recommendation_counts):
    arr = np.sort(np.array(recommendation_counts))
    if np.sum(arr) == 0: return 0.0
    n = len(arr)
    return (n + 1 - 2 * np.sum(np.cumsum(arr)) / np.sum(arr)) / n

def compute_metrics(model, adj_dict, test_df, train_df, user_map, item_map, 
                    user_labels, movie_tiers, K=100):
    model.eval()
    with torch.no_grad():
        final_emb = model(adj_dict)
        
        u_nodes = list(user_map.values())
        i_nodes = list(item_map.values())
        
        valid_test = test_df[test_df['UserID'].isin(user_map.keys()) & 
                             test_df['MovieID'].isin(item_map.keys())].copy()
        if valid_test.empty: return None, 0

        valid_test['u_node'] = valid_test['UserID'].map(user_map)
        valid_test['i_node'] = valid_test['MovieID'].map(item_map)
        
        user_gt = valid_test.groupby('u_node')['i_node'].apply(set).to_dict()
        test_u_nodes = list(user_gt.keys())
        
        test_u_tensor = torch.LongTensor(test_u_nodes).to(device)
        all_i_tensor = torch.LongTensor(i_nodes).to(device)
        
        user_embs = final_emb[test_u_tensor]
        item_embs = final_emb[all_i_tensor]
        
        scores = torch.matmul(user_embs, item_embs.t())
        
        train_interactions = train_df.groupby('UserID')['MovieID'].apply(list).to_dict()
        local_idx_to_node_id = {idx: node_id for idx, node_id in enumerate(i_nodes)}
        node_to_u_id_map = {v: k for k, v in user_map.items()}
        
        for idx, u_node in enumerate(test_u_nodes):
            u_id = node_to_u_id_map[u_node]
            if u_id in train_interactions:
                raw_seen = train_interactions[u_id]
                seen_nodes = set(item_map[m] for m in raw_seen if m in item_map)
                mask_indices = [k for k, v in local_idx_to_node_id.items() if v in seen_nodes]
                if mask_indices:
                    scores[idx, mask_indices] = -1e9

        _, top_indices = torch.topk(scores, K, dim=1)
        top_indices = top_indices.cpu().numpy()
        
        recall_arr, ndcg_arr = [], []
        group_recalls = {'mainstream': [], 'niche': []}
        pop_counts = {'mainstream': {'high': 0, 'medium': 0, 'low': 0}, 
                      'niche': {'high': 0, 'medium': 0, 'low': 0}}
        total_recs = {'mainstream': 0, 'niche': 0}
        
        node_to_m_id = {v: k for k, v in item_map.items()}
        all_item_counts = np.zeros(len(i_nodes))

        for idx, u_node in enumerate(test_u_nodes):
            truth = user_gt[u_node]
            rec_node_ids = [local_idx_to_node_id[x] for x in top_indices[idx]]
            
            hits = len(set(rec_node_ids) & truth)
            recall_arr.append(hits / len(truth))
            
            dcg, idcg = 0.0, 0.0
            for k, r in enumerate(rec_node_ids):
                if r in truth: dcg += 1.0 / np.log2(k + 2)
            for k in range(min(len(truth), K)): idcg += 1.0 / np.log2(k + 2)
            ndcg_arr.append(dcg/idcg if idcg > 0 else 0)
            
            u_id = node_to_u_id_map[u_node]
            u_type = user_labels.get(u_id, 'niche')
            group_recalls[u_type].append(hits / len(truth))
            total_recs[u_type] += K
            
            for local_idx, r_node in zip(top_indices[idx], rec_node_ids):
                all_item_counts[local_idx] += 1
                raw_mid = node_to_m_id.get(r_node)
                tier = movie_tiers.get(raw_mid, 'low')
                pop_counts[u_type][tier] += 1
        
        raw_exp = {grp: {t: (pop_counts[grp][t]/total_recs[grp]) if total_recs[grp]>0 else 0.0 for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        formatted_pop = {grp: {t: f"{(raw_exp[grp][t]*100):.2f}%" for t in ['low','medium','high']} for grp in ['mainstream','niche']}
        
        eo = abs(np.mean(group_recalls['mainstream']) - np.mean(group_recalls['niche'])) if group_recalls['mainstream'] else 0
        dp = (abs(raw_exp['mainstream']['low'] - raw_exp['niche']['low']) +
              abs(raw_exp['mainstream']['medium'] - raw_exp['niche']['medium']) +
              abs(raw_exp['mainstream']['high'] - raw_exp['niche']['high'])) / 3.0
        gini_val = get_gini(all_item_counts)
        
        return {
            'Recall': np.mean(recall_arr), 'NDCG': np.mean(ndcg_arr), 'EO': eo, 'DP': dp, 
            'Gini': gini_val, 'PopExposure': formatted_pop, 'RawExposure': raw_exp
        }, len(test_u_nodes)

# ----------------------------
# 3. Model Logic: Multi-Relational DistMult (Baseline)
# ----------------------------
class DistMultModel(nn.Module):
    def __init__(self, num_nodes, num_relations, embedding_dim=64):
        super().__init__()
        self.node_emb = nn.Embedding(num_nodes, embedding_dim)
        self.rel_emb = nn.Embedding(num_relations, embedding_dim)
        nn.init.xavier_uniform_(self.node_emb.weight)
        nn.init.xavier_uniform_(self.rel_emb.weight)

    def forward(self, adj_dict=None):
        h = self.node_emb.weight
        r_0 = self.rel_emb.weight[0] 
        return h * r_0

    def get_triple_score(self, h_idx, r_idx, t_idx):
        h = self.node_emb(h_idx)
        r = self.rel_emb(r_idx)
        t = self.node_emb(t_idx)
        return (h * r * t).sum(dim=1)

def build_relational_adjacency(kg_edges_df, num_nodes):
    return {}

# ----------------------------
# 4. Training Loop (Baseline: No Fairness Retries)
# ----------------------------
LR, EPOCHS, BATCH_SIZE = 0.001, 50, 2048
KG_LAMBDA = 0.1 

prev_model_state = None
prev_node_map = {} 

print(f"{'Slice Pair':<13} | {'Users':<6} | {'Recall':<8} | {'NDCG':<8} | {'EO':<8} | {'DP':<8} | {'Gini':<8}")
print("-" * 105)

slice_nums = sorted(ratings['SliceNum'].unique())
for s in slice_nums[:-1]: 
    train_df = ratings[ratings['SliceNum'] == s].copy()
    test_df  = ratings[ratings['SliceNum'] == s+1].copy()
    
    user_map_df = pd.read_csv(f"slice_metadata/kg/user_map_slice_{s}.csv")
    movie_map_df = pd.read_csv(f"slice_metadata/kg/movie_map_slice_{s}.csv")
    kg_edges_df = pd.read_csv(f"slice_metadata/kg/kg_edges_slice_{s}.csv")
    
    u_map = dict(zip(user_map_df.UserID, user_map_df.NodeID))
    i_map = dict(zip(movie_map_df.MovieID, movie_map_df.NodeID))
    
    current_node_lookup = {}
    for k, v in u_map.items(): current_node_lookup[f"u_{k}"] = v
    for k, v in i_map.items(): current_node_lookup[f"m_{k}"] = v

    user_labels = pd.read_csv(f"slice_metadata/user_labels_slice_{s}.csv").set_index('UserID')['UserType'].to_dict()
    movie_tiers = pd.read_csv(f"slice_metadata/movie_pop_tiers_slice_{s}.csv").set_index('MovieID')['PopTier'].to_dict()

    max_node_id = max(kg_edges_df['Src'].max(), kg_edges_df['Dst'].max()) + 1
    max_rel_id = kg_edges_df['RelType'].max() + 1
    
    adj_dict = build_relational_adjacency(kg_edges_df, max_node_id)
    
    kg_h = torch.LongTensor(kg_edges_df['Src'].values).to(device)
    kg_r = torch.LongTensor(kg_edges_df['RelType'].values).to(device)
    kg_t = torch.LongTensor(kg_edges_df['Dst'].values).to(device)
    num_kg_triples = len(kg_h)

    train_df = train_df[train_df['UserID'].isin(u_map) & train_df['MovieID'].isin(i_map)]
    train_user_tensor = torch.LongTensor(train_df['UserID'].map(u_map).values).to(device)
    train_item_tensor = torch.LongTensor(train_df['MovieID'].map(i_map).values).to(device)
    num_train_samples = len(train_user_tensor)

    # BASELINE: No weighting based on tiers or user types
    # All weights are set to 1.0
    weight_matrix = torch.ones((2, 3), dtype=torch.float, device=device)
    
    tier_idx_map = {'low': 0, 'medium': 1, 'high': 2}
    u_type_idx_map = {'mainstream': 0, 'niche': 1}
    
    item_tier_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for mid, node_id in i_map.items():
        item_tier_tensor[node_id] = tier_idx_map.get(movie_tiers.get(mid, 'low'), 0)

    user_type_tensor = torch.zeros(max_node_id, dtype=torch.long, device=device)
    for uid, node_id in u_map.items():
        user_type_tensor[node_id] = u_type_idx_map.get(user_labels.get(uid, 'niche'), 1)

    # ----------------------------
    # WARM STARTING
    # ----------------------------
    model = DistMultModel(max_node_id, max_rel_id).to(device)
    if prev_model_state:
        with torch.no_grad():
            prev_nodes = prev_model_state['node_emb.weight'].to(device)
            for entity_key, curr_id in current_node_lookup.items():
                if entity_key in prev_node_map:
                    prev_id = prev_node_map[entity_key]
                    if prev_id < prev_nodes.shape[0]:
                        model.node_emb.weight.data[curr_id] = prev_nodes[prev_id]
            
            if 'rel_emb.weight' in prev_model_state:
                prev_rel = prev_model_state['rel_emb.weight'].to(device)
                sz = min(model.rel_emb.weight.shape[0], prev_rel.shape[0])
                model.rel_emb.weight.data[:sz] = prev_rel[:sz]

    optimizer = optim.Adam(model.parameters(), lr=LR)
    model.train()
    all_item_nodes = torch.LongTensor(list(i_map.values())).to(device)
    
    # Single training pass per slice (no retries)
    for epoch in range(EPOCHS):
        rand_indices = torch.randint(0, len(all_item_nodes), (num_train_samples,), device=device)
        neg_nodes = all_item_nodes[rand_indices]
        kg_neg_nodes = torch.randint(0, max_node_id, (num_train_samples,), device=device)
        
        idxs = torch.randperm(num_train_samples, device=device)
        for i in range(0, num_train_samples, BATCH_SIZE):
            b = idxs[i:i+BATCH_SIZE]
            u_b, p_b, n_b = train_user_tensor[b], train_item_tensor[b], neg_nodes[b]
            
            # Recommendation Loss
            final_emb = model(adj_dict)
            pos_scores = (final_emb[u_b] * final_emb[p_b]).sum(1)
            neg_scores = (final_emb[u_b] * final_emb[n_b]).sum(1)
            bpr = -torch.nn.functional.logsigmoid(pos_scores - neg_scores).mean()

            # KG Loss
            kg_idx = torch.randint(0, num_kg_triples, (len(b),), device=device)
            h_k, r_k, t_k = kg_h[kg_idx], kg_r[kg_idx], kg_t[kg_idx]
            t_neg_k = kg_neg_nodes[b]
            pos_kg = model.get_triple_score(h_k, r_k, t_k)
            neg_kg = model.get_triple_score(h_k, r_k, t_neg_k)
            kg_loss = -torch.nn.functional.logsigmoid(pos_kg - neg_kg).mean()

            loss = bpr + (KG_LAMBDA * kg_loss)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    # Final Metrics for slice
    metrics, count = compute_metrics(model, adj_dict, test_df, train_df, u_map, i_map, user_labels, movie_tiers)
    
    if metrics:
        print(f"{s} -> {s+1:<8} | {count:<6} | {metrics['Recall']:.4f}    | {metrics['NDCG']:.4f}    | {metrics['EO']:.4f}    | {metrics['DP']:.4f}    | {metrics['Gini']:.4f}")
        print(f"PopExposure: {metrics['PopExposure']} (Baseline - No Weights)")
        
        # Save state for next slice's warm start
        prev_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        prev_node_map = current_node_lookup.copy()

Using device: cuda
Slice Pair    | Users  | Recall   | NDCG     | EO       | DP       | Gini    
---------------------------------------------------------------------------------------------------------
1 -> 2        | 96     | 0.2346    | 0.1781    | 0.0787    | 0.1737    | 0.8029
PopExposure: {'mainstream': {'low': '0.13%', 'medium': '5.55%', 'high': '94.32%'}, 'niche': {'low': '5.02%', 'medium': '26.71%', 'high': '68.27%'}} (Baseline - No Weights)
2 -> 3        | 117    | 0.2061    | 0.1375    | 0.0826    | 0.1621    | 0.7475
PopExposure: {'mainstream': {'low': '0.51%', 'medium': '10.56%', 'high': '88.93%'}, 'niche': {'low': '6.51%', 'medium': '28.86%', 'high': '64.62%'}} (Baseline - No Weights)
3 -> 4        | 166    | 0.1996    | 0.1176    | 0.0571    | 0.1398    | 0.6771
PopExposure: {'mainstream': {'low': '5.35%', 'medium': '18.76%', 'high': '75.89%'}, 'niche': {'low': '13.64%', 'medium': '31.43%', 'high': '54.92%'}} (Baseline - No Weights)
4 -> 5        | 211    | 0.1927    | 0